# Programming Assignment 3

**Colab Free Tier has pretty strict GPU usage time limits**

**I would HIGHLY recommend doing all development on the CPU runtime instance (you can change this by clicking the dropdown arrow next to Connect (top right) --> Change Runtime Type --> CPU or T4), and then only switching to the T4 runtime (T4 is an Nvidia GPU), when you are ready to actually train the model**

## 1. Setup

Installing packages and downloading Atari environments

In [ ]:
pip install ale-py

In [ ]:
!pip install git+https://github.com/MLivanos/colabgymrender.git

### Visualize

In [ ]:
import base64
import os
from IPython.display import HTML, display

def render_mp4_video(video_folder):
    """Finds the first mp4 in the folder and displays it in the notebook."""
    video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]

    if not video_files:
        print("No video found! Make sure the episode actually finished and triggered the recorder.")
        return

    # Pick the most recent video
    video_path = os.path.join(video_folder, sorted(video_files)[-1])

    video_file = open(video_path, "rb").read()
    encoded = base64.b64encode(video_file).decode("ascii")

    html_code = f"""
    <video width="600" height="400" controls>
        <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """
    display(HTML(html_code))

### Demo of environment

In [ ]:
import gymnasium as gym
import ale_py

# 1. Setup Environment
gym.register_envs(ale_py)
env = gym.make("ALE/MsPacman-v5", render_mode="rgb_array")

# 2. Wrap for recording (recording the first episode)
video_path = "./saved-videos"
env = gym.wrappers.RecordVideo(
    env,
    video_folder=video_path,
    episode_trigger=lambda x: x == 0,
    name_prefix="pacman-run"
)

# 3. Run a quick episode
obs, info = env.reset()
done = False
while not done:
    action = env.action_space.sample() # Random actions
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

env.close()

# 4. SHOW THE VIDEO IN THE CELL
render_mp4_video(video_path)

## Deep Q-Learning

### 2.1 Environment and Model Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os
from collections import deque
import gymnasium as gym
import ale_py
import matplotlib.pyplot as plt
import time

# Gymnasium v1.0.0+ standard imports
from gymnasium.wrappers import (
    AtariPreprocessing,
    FrameStackObservation,
    TransformObservation,
    RecordVideo
)

def make_lite_env(env_id, render_mode=None, is_eval=False):
    """
    Standardized Atari Environment Factory.

    This function applies the 'Atari Preprocessing' suite which is essential
    for training stability. It reduces memory usage, simplifies the input
    space, and provides the agent with 'vision' of movement.
    """

    # 1. Registration: Ensures the ALE (Arcade Learning Environment) environments
    # are recognized by Gymnasium.
    gym.register_envs(ale_py)

    # 2. Base Env: Creates the raw environment. 'frameskip=1' is set here because
    # we handle skipping manually in the AtariPreprocessing wrapper for finer control.
    env = gym.make(env_id, render_mode=render_mode, frameskip=1)

    # 3. AtariPreprocessing: The "Workhorse" wrapper.
    # - screen_size=84: Downscales from ~210x160 to 84x84 (drastic speedup).
    # - grayscale_obs: Removes color (RGB) to reduce input channels from 3 to 1.
    # - frame_skip=4: The agent only 'sees' every 4th frame. The chosen action
    #   is repeated for the skipped frames. This speeds up training 4x.
    # - terminal_on_life_loss: CRITICAL FOR TRAINING. In many Atari games,
    #   losing a life doesn't reset the game, but we want the agent to treat
    #   it as a 'Failure' state to learn survival. During evaluation (is_eval),
    #   we set this to False to see the agent's full 3-life performance.
    env = AtariPreprocessing(
        env,
        screen_size=84,
        grayscale_obs=True,
        frame_skip=4,
        terminal_on_life_loss=not is_eval
    )

    # 4. FrameStack: Adds temporal context (Time).
    # A single grayscale frame is just a still image; the agent can't tell if
    # Pac-Man is moving left or right. By stacking the last 4 frames, the CNN
    # can see 'velocity' and 'direction' of ghosts and players.
    # Input shape becomes: (4, 84, 84)
    env = FrameStackObservation(env, stack_size=4)

    new_obs_space = gym.spaces.Box(
            low=0, high=255, shape=env.observation_space.shape, dtype=np.uint8
        )
    env = TransformObservation(
        env,
        lambda obs: np.array(obs).astype(np.uint8),
        new_obs_space
    )

    return env

Mount the Google Drive

In case the runtime disconnected, temporary files lose.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Plot the loss and reward

In [ ]:
def plot_metrics(rewards_history, losses, reward_window=20):
    """
    X-axis = Training Steps (matching console logs)
    Y-axis = Episode Rewards
    """
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    if len(rewards_history) > 0:
        # 1. Convert to numpy array: col 0 is Steps, col 1 is Reward
        data = np.array(rewards_history)
        steps = data[:, 0]
        rewards = data[:, 1]

        # 2. Reward Plot
        ax[0].plot(steps, rewards, label='Episode Score', alpha=0.3, color='dodgerblue')

        # Calculate moving average
        if len(rewards) > reward_window:
            ma = np.convolve(rewards, np.ones(reward_window)/reward_window, mode='valid')
            ma_steps = steps[reward_window-1:]
            ax[0].plot(ma_steps, ma, label=f'{reward_window}-Ep Moving Avg', color='red', linewidth=2)

        ax[0].set_title("Ms. Pacman Performance")
        ax[0].set_xlabel("Total Steps")
        ax[0].set_ylabel("Reward")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

    # 3. Loss Plot (Logged per optimization step)
    if len(losses) > 100:
        ma_loss = np.convolve(losses, np.ones(100)/100, mode='valid')
        ax[1].plot(ma_loss, color='forestgreen', label='Loss Trend')
    else:
        ax[1].plot(losses, color='forestgreen', alpha=0.5)

    ax[1].set_title("Training Loss (Internal Error)")
    ax[1].set_xlabel("Optimization Steps")
    ax[1].set_ylabel("MSE Loss")
    ax[1].set_yscale('log') # Log scale handles the huge drops in loss better
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_progress.png')
    plt.show()

### 2.2 Q Network

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(4, 16, 8, stride=4),  # (4, 84, 84) -> (16, 20, 20)
            nn.ReLU(),
            nn.Conv2d(16, 32, 4, stride=2), # (16, 20, 20) -> (32, 9, 9)
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 9 * 9, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions)
        )

    def forward(self, x):
      x = x.float() / 255.0
      return self.network(x)

### 2.3 Configuaration and hyperparameters

In [ ]:
# --- SYSTEM & ENVIRONMENT ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Hardware: GPU if available, else CPU
ENV_ID = "ALE/MsPacman-v5"                                           # Game: Ms. Pacman (v5 includes sticky actions)

# --- HYPERPARAMETERS ---
LR = 1e-4         # Learning Rate: Step size for weight updates
BATCH_SIZE = 32      # Minibatch: Number of transitions used per training step
GAMMA = 0.99         # Discount Factor: Importance of future vs. immediate rewards

# --- TRAINING SCALE ---
MEMORY_SIZE = 100000  # Replay Buffer: Max transitions stored for experience replay
TOTAL_STEPS = 500000 # Training Duration: Total frames the agent will experience

CHECKPOINT_PATH = "/content/drive/MyDrive/dqn_pacman.pth"

history_rewards = []  # Total reward per episode/life
history_loss = []     # MSE loss per optimization step
current_episode_reward = 0


### 2.4 Initialization

In [ ]:
env = make_lite_env(ENV_ID, is_eval=False) # Change is_eval to True and the reward in the training loop to train on all three lives.
q_net = QNetwork(env.action_space.n).to(DEVICE)
target_net = QNetwork(env.action_space.n).to(DEVICE)
optimizer = optim.Adam(q_net.parameters(), lr=LR)
memory = deque(maxlen=MEMORY_SIZE)

start_step = 0
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    q_net.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_step = checkpoint['step']
    # Load history if you want to keep plots continuous
    history_rewards = checkpoint.get('history_rewards', [])
    history_loss = checkpoint.get('history_loss', [])
    print(f"Resuming from step {start_step}")

target_net.load_state_dict(q_net.state_dict())
obs, _ = env.reset()

### 2.5 Training Loop

In [ ]:
EPSILON_DECAY_STEPS = 250000 
EPSILON_MIN = 0.05

start_time = time.time()
print(f"Starting training on {DEVICE}...")
for step in range(start_step, TOTAL_STEPS):
    epsilon = max(EPSILON_MIN, 1.0 - (step / EPSILON_DECAY_STEPS))

    if random.random() < epsilon:
        action = env.action_space.sample()
    else:
        with torch.no_grad():
            state_t = torch.FloatTensor(np.array([obs])).to(DEVICE) #
            q_values = q_net(state_t)
            action = torch.argmax(q_values, dim=1).item()

    next_obs, reward, terminated, truncated, _ = env.step(action)
    current_episode_reward += reward
    memory.append((obs, action, reward, next_obs, terminated))
    obs = next_obs

    if terminated or truncated:
        history_rewards.append([step, current_episode_reward])
        current_episode_reward = 0
        obs, _ = env.reset()

    if len(memory) > BATCH_SIZE:
        # Sample BATCH_SIZE experiences from the replay buffer
        batch = random.sample(memory, BATCH_SIZE)
        s, a, r, ns, d = zip(*batch)
        s = torch.from_numpy(np.array(s)).to(DEVICE)
        ns = torch.from_numpy(np.array(ns)).to(DEVICE)

        a = torch.LongTensor(a).to(DEVICE).unsqueeze(1)
        r = torch.FloatTensor(r).to(DEVICE)
        d = torch.FloatTensor(d).to(DEVICE)

        # Now when you call q_net(s), the normalization happens on GPU!
        current_q = q_net(s).gather(1, a).squeeze()

        with torch.no_grad():
            next_q = target_net(ns).max(1)[0]
        # target_q = reward + (discount_factor * max_next_q * (1 - done))
        target_q = r + (GAMMA * next_q * (1 - d))

        # MSE Loss: mean of squared differences between target and prediction
        # loss = torch.nn.functional.mse_loss(current_q, target_q)
        loss = ((target_q - current_q) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history_loss.append(loss.item())

    if step % 10000 == 0 and step > start_step:
        target_net.load_state_dict(q_net.state_dict())
        torch.save({
            'step': step,
            'model_state_dict': q_net.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history_rewards': history_rewards,
            'history_loss': history_loss
        }, CHECKPOINT_PATH)

        # Extract just the rewards (column 1) from the last 20 entries
        last_rewards = [r[1] for r in history_rewards[-20:]]
        avg_reward = np.mean(last_rewards) if last_rewards else 0
        print(f"Step {step} | Eps: {epsilon:.2f} | Avg Reward (Last 20): {avg_reward:.1f}")

env.close()

#Calculate and display execution time
end_time = time.time()
total_duration = end_time - start_time
minutes = int(total_duration // 60)
seconds = int(total_duration % 60)

print(f"\nTraining Complete!")
print(f"Total Training Time: {minutes}m {seconds}s")

### 2.6 Training Results

Plot

In [ ]:
# Call the plot function
# This uses the history collected during the loop
plot_metrics(history_rewards, history_loss)

Game Video

In [ ]:
eval_env = make_lite_env(ENV_ID, render_mode="rgb_array", is_eval=True)
video_path = "./pacman_trained"
eval_env = RecordVideo(eval_env, video_folder=video_path, name_prefix="dqn-pacman", episode_trigger=lambda x: True)

obs, _ = eval_env.reset()
done = False
total_eval_score = 0  # <--- NEW: Score tracker

while not done:
    with torch.no_grad():
        state_t = torch.FloatTensor(np.array([obs])).to(DEVICE)
        action = q_net(state_t).argmax().item()

    obs, reward, terminated, truncated, _ = eval_env.step(action)
    total_eval_score += reward  # <--- Accumulate score
    done = terminated or truncated

print(f"--- Evaluation Complete ---")
print(f"Final Game Score: {total_eval_score}") # <--- Print the result
eval_env.close()

render_mp4_video(video_path)

### 2.7 Save the model if needed

In [ ]:
# --- After the training loop finishes ---

final_model_path = "/content/drive/MyDrive/pacman_final_dqn.pth"

# 1. Save the model weights and training metadata
save_data = {
    'model_state_dict': q_net.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'total_steps': TOTAL_STEPS,
    'epsilon_final': epsilon,
    'env_id': ENV_ID
}

torch.save(save_data, final_model_path)
print(f"\n✅ Final model saved to {final_model_path}")

## 3. Bonus Points Algorithm

Game score should be higher than 1500 to be eligible for bonus points.

Print the reward and score. Show and download the video.